# Financial Document Intelligence & Evidence-Grounded RAG System

**Architecture:** Multimodal PDF Parsing → Modality-Aware Chunking → Dense (BGE) + BM25 → RRF Hybrid Retrieval → Cross-Encoder Reranking → Grounded Generation (GPT-4o-mini) → Numerical Verification → Ablation Study

**This notebook covers:**
1. Setup & dependency installation
2. Document download
3. Ingestion pipeline (text + tables + charts)
4. Chunking
5. Building FAISS + BM25 indexes
6. Query classification demo
7. Hybrid retrieval + reranking demo
8. Grounded generation with citations
9. Numerical verification demo
10. Full evaluation with ablation study

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────
!pip install -q pymupdf pdfplumber pillow sentence-transformers faiss-cpu rank-bm25 \
    openai transformers torch tqdm pyyaml loguru jsonlines ragas scikit-learn

# Clone the project repo (adjust URL to your GitHub)
# !git clone https://github.com/YOUR_USERNAME/financial-rag.git
# %cd financial-rag

print('✓ Dependencies installed')

In [ ]:
# ── Cell 2: Setup ─────────────────────────────────────────────────────
import os
import json
import yaml
import logging
from pathlib import Path

# Set your OpenAI key here or via Colab secrets
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')  # or: = 'sk-...'

# Create directories
for d in ['data/raw', 'data/processed/chart_images', 'data/indexes', 'data/results']:
    Path(d).mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
print('✓ Setup complete')

In [ ]:
# ── Cell 3: Download financial reports ────────────────────────────────
# For the demo, we'll use one report.  In production, download all 5.
# Tesla 10-K FY2023 (public EDGAR filing)
import urllib.request

DOCS = [
    {
        'company': 'Tesla',
        'year': 2023,
        'type': '10-K',
        'filename': 'Tesla_10K_2023.pdf',
        'url': 'https://ir.tesla.com/_flysystem/s3/sec/000095017024020448/tsla-20231231.pdf'
    },
]

for doc in DOCS:
    path = f"data/raw/{doc['filename']}"
    if not Path(path).exists():
        print(f"Downloading {doc['filename']}...")
        urllib.request.urlretrieve(doc['url'], path)
    else:
        print(f"Already exists: {doc['filename']}")

print('✓ Documents ready')

In [ ]:
# ── Cell 4: PDF Ingestion ─────────────────────────────────────────────
from src.ingestion.pdf_parser import FinancialPDFParser

parser = FinancialPDFParser(
    pdf_path='data/raw/Tesla_10K_2023.pdf',
    company='Tesla',
    document='Tesla 10-K 2023',
    year=2023,
    dpi=150,
)

elements = parser.parse()

# Summary
from collections import Counter
type_counts = Counter(e.element_type for e in elements)
print(f"\nExtracted elements:")
for etype, count in type_counts.items():
    print(f"  {etype}: {count}")

# Show a text sample
text_samples = [e for e in elements if e.element_type == 'text'][:2]
for s in text_samples:
    print(f"\nPage {s.page} | Section: {s.section or 'N/A'}")
    print(s.content[:300])

In [ ]:
# ── Cell 5: Chart Description ─────────────────────────────────────────
import openai
from src.ingestion.chart_describer import ChartDescriber
from PIL import Image

client = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])
describer = ChartDescriber(openai_client=client, model='gpt-4o-mini')

# Find chart elements and save their images
chart_elements = []
for elem in elements:
    if elem.element_type == 'chart' and isinstance(elem.content, Image.Image):
        img_path = f"data/processed/chart_images/tesla_{elem.page}.png"
        elem.content.save(img_path)
        chart_elements.append({
            'element_type': 'chart',
            'content': img_path,
            'company': elem.company,
            'document': elem.document,
            'year': elem.year,
            'page': elem.page,
            'section': elem.section,
            'caption': elem.caption,
        })

print(f"Found {len(chart_elements)} chart images")

# Describe first 3 charts (cost-saving demo)
described = describer.describe_batch(chart_elements[:3])

if described:
    print("\nExample chart description:")
    print(described[0].get('description', 'No description'))[:500]

In [ ]:
# ── Cell 6: Modality-Aware Chunking ───────────────────────────────────
from src.ingestion.chunker import Chunker

# Prepare all elements as dicts
all_element_dicts = []
for e in elements:
    d = e.__dict__.copy()
    if hasattr(d.get('content'), 'save'):  # PIL Image — skip, handled as chart_elements
        continue
    all_element_dicts.append(d)

# Add described charts
all_element_dicts.extend(described)

chunker = Chunker(text_max_tokens=512, text_overlap=64, table_max_tokens=1024)
chunks = chunker.chunk(all_element_dicts)

from collections import Counter
ctype_counts = Counter(c.chunk_type for c in chunks)
print(f"\nTotal chunks: {len(chunks)}")
for ctype, count in ctype_counts.items():
    print(f"  {ctype}: {count}")

# Show examples
print("\n── Example text chunk ──")
tc = next((c for c in chunks if c.chunk_type == 'text'), None)
if tc: print(tc.content[:400])

print("\n── Example table chunk ──")
tabc = next((c for c in chunks if c.chunk_type == 'table'), None)
if tabc: print(tabc.content[:500])

print("\n── Example chart chunk ──")
cc = next((c for c in chunks if c.chunk_type == 'chart'), None)
if cc: print(cc.content[:500])

In [ ]:
# ── Cell 7: Build Dense + BM25 Indexes ───────────────────────────────
from src.embeddings.dense_embedder import DenseEmbedder
from src.retrieval.bm25_index import BM25Index

chunk_dicts = [c.to_dict() for c in chunks]

# FAISS
print("Building FAISS dense index...")
embedder = DenseEmbedder(
    model_name='BAAI/bge-base-en-v1.5',
    device='cpu',
    batch_size=32,
    index_path='data/indexes/faiss.index',
    metadata_path='data/indexes/faiss_metadata.jsonl',
)
embedder.build_index(chunk_dicts)

# BM25
print("\nBuilding BM25 index...")
bm25_idx = BM25Index(index_path='data/indexes/bm25.pkl')
bm25_idx.build(chunk_dicts)
bm25_idx.save()

print("\n✓ Both indexes built")

In [ ]:
# ── Cell 8: Query Classification Demo ────────────────────────────────
from src.retrieval.query_classifier import QueryClassifier

clf = QueryClassifier()

test_queries = [
    "What was Tesla's total revenue in FY2023?",
    "How did vehicle deliveries change from Q2 to Q3?",
    "What factors contributed to revenue growth?",
    "Compare Tesla and Nvidia revenue growth.",
    "How has the gross margin trended over time?",
]

print(f"{'Query':<55} {'Type':<15} {'Dense':>6} {'BM25':>6}")
print('-' * 85)
for q in test_queries:
    r = clf.classify(q)
    print(f"{q[:54]:<55} {r.query_type.value:<15} {r.dense_weight:>6.0%} {r.bm25_weight:>6.0%}")

In [ ]:
# ── Cell 9: Hybrid Retrieval + Reranking Demo ─────────────────────────
from src.retrieval.hybrid_retriever import HybridRetriever
from src.retrieval.reranker import CrossEncoderReranker

# Load indexes
embedder.load_index()
bm25_idx.load()

retriever = HybridRetriever(
    dense_embedder=embedder,
    bm25_index=bm25_idx,
    dense_top_k=20,
    bm25_top_k=20,
    rrf_k=60,
    use_classifier=True,
)

reranker = CrossEncoderReranker(
    model_name='BAAI/bge-reranker-base',
    device='cpu',
    top_k=5,
)

query = "What was Tesla's gross margin and revenue for FY2023?"
print(f"Query: {query}\n")

# Dense only
dense_results = retriever.retrieve_dense_only(query, top_k=5)
print("== Dense Only (top 5) ==")
for i, (c, s) in enumerate(dense_results, 1):
    print(f"{i}. [{c['chunk_type']}] p{c['page']} score={s:.3f}: {c['content'][:80]}...")

# Hybrid
hybrid_results = retriever.retrieve(query, top_k=20)
print(f"\n== Hybrid RRF (top 20 → rerank to top 5) ==")
reranked = reranker.rerank(query, hybrid_results)
for i, (c, s) in enumerate(reranked, 1):
    print(f"{i}. [{c['chunk_type']}] p{c['page']} rerank={s:.3f}: {c['content'][:80]}...")

In [ ]:
# ── Cell 10: Grounded Generation Demo ────────────────────────────────
from src.generation.grounded_generator import GroundedGenerator

generator = GroundedGenerator(
    openai_client=client,
    model='gpt-4o-mini',
    max_tokens=1024,
    enable_verification=True,
)

result = generator.generate(query, reranked)

print("ANSWER:")
print(result.answer)
print(f"\nConfidence: {result.confidence}")
print(f"Abstained: {result.abstained}")

if result.citations:
    print("\nCITATIONS:")
    for c in result.citations:
        print(f"  {c.get('company')} | {c.get('document')} | Page {c.get('page')}")

if result.verification:
    print(f"\nNUMERICAL VERIFICATION: {result.verification.summary}")

In [ ]:
# ── Cell 11: Numerical Verification Deep Dive ─────────────────────────
from src.generation.numerical_verifier import NumericalVerifier, extract_numbers

verifier = NumericalVerifier(tolerance=0.02)

# Simulate a hallucinated answer
hallucinated_answer = "Tesla's revenue was $97.3B in FY2023 with a gross margin of 21.5%."
correct_evidence = "Tesla total revenues were $96.8 billion for fiscal year 2023. Gross margin was 18.2%."

print("Answer:", hallucinated_answer)
print("Evidence:", correct_evidence)
print()

report = verifier.verify(hallucinated_answer, [correct_evidence])
print(f"Verified: {report.verified}")
print(f"Summary: {report.summary}")
if report.mismatches:
    print("\nMismatches:")
    for m in report.mismatches:
        print(f"  ⚠️ {m.message}")

In [ ]:
# ── Cell 12: Out-of-Scope Abstention Test ─────────────────────────────
oos_queries = [
    "What was Google's revenue in FY2023?",
    "How many employees does Samsung have?",
]

for q in oos_queries:
    print(f"Query: {q}")
    candidates = retriever.retrieve(q, top_k=20)
    final = reranker.rerank(q, candidates)
    result = generator.generate(q, final)
    status = '✅ ABSTAINED' if result.abstained else '❌ DID NOT ABSTAIN'
    print(f"  {status}: {result.answer[:150]}")
    print()

In [ ]:
# ── Cell 13: Retrieval Evaluation (Ablation Study) ────────────────────
from src.evaluation.metrics import compute_retrieval_metrics, aggregate_metrics
from src.evaluation.test_set import TEST_SET

def chunks_to_id_map(chunk_dicts):
    return {c['chunk_id']: c for c in chunk_dicts}

def gold_ids_from_question(q, chunk_dicts):
    pages = q.get('evidence_pages', [])
    company = q.get('company', '').lower()
    return {
        c['chunk_id'] for c in chunk_dicts
        if c.get('company', '').lower() == company
        and c.get('page', -1) in pages
    }

# Run ablation on a subset (in-scope only)
in_scope = [q for q in TEST_SET if q['in_scope'] and q['evidence_pages']]
print(f"Evaluating on {len(in_scope)} in-scope questions with evidence pages...")

ablation_configs = {
    'Dense only': lambda q: retriever.retrieve_dense_only(q, top_k=20),
    'BM25 only': lambda q: retriever.retrieve_bm25_only(q, top_k=20),
    'Hybrid (RRF)': lambda q: retriever.retrieve(q, top_k=20),
    'Hybrid + Rerank': lambda q: reranker.rerank(q, retriever.retrieve(q, top_k=20)),
}

ablation_results = {}
for cfg_name, retrieve_fn in ablation_configs.items():
    per_q = []
    for q in in_scope[:10]:  # limit to 10 for speed in demo
        results = retrieve_fn(q['question'])
        retrieved_ids = [c['chunk_id'] for c, _ in results]
        gold = gold_ids_from_question(q, chunk_dicts)
        if gold:
            per_q.append(compute_retrieval_metrics(retrieved_ids, gold))
    ablation_results[cfg_name] = aggregate_metrics(per_q)

# Print table
print(f"\n{'Config':<25} {'R@1':>5} {'R@3':>5} {'R@5':>5} {'NDCG@5':>7} {'MRR':>6}")
print('-' * 55)
for cfg_name, metrics in ablation_results.items():
    print(
        f"{cfg_name:<25}"
        f" {metrics.get('recall@1', 0):>5.3f}"
        f" {metrics.get('recall@3', 0):>5.3f}"
        f" {metrics.get('recall@5', 0):>5.3f}"
        f" {metrics.get('ndcg@5', 0):>7.3f}"
        f" {metrics.get('mrr', 0):>6.3f}"
    )

In [ ]:
# ── Cell 14: Plot Ablation Results ───────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

configs = list(ablation_results.keys())
recall5 = [ablation_results[c].get('recall@5', 0) for c in configs]
ndcg5   = [ablation_results[c].get('ndcg@5', 0) for c in configs]
mrr     = [ablation_results[c].get('mrr', 0) for c in configs]

x = np.arange(len(configs))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width, recall5, width, label='Recall@5', color='#2196F3', alpha=0.85)
bars2 = ax.bar(x,         ndcg5,   width, label='NDCG@5',  color='#4CAF50', alpha=0.85)
bars3 = ax.bar(x + width, mrr,     width, label='MRR',     color='#FF9800', alpha=0.85)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Retrieval Ablation Study — Financial RAG', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(configs, fontsize=10)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f'{h:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('data/results/ablation_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to data/results/ablation_plot.png')

In [ ]:
# ── Cell 15: End-to-End Q&A Demo ─────────────────────────────────────
def ask(question: str):
    print(f"\n{'='*70}")
    print(f"Q: {question}")
    print('='*70)

    # Classify
    clf_result = clf.classify(question)
    print(f"[Type: {clf_result.query_type.value} | Dense {clf_result.dense_weight:.0%} / BM25 {clf_result.bm25_weight:.0%}]")

    # Retrieve + rerank
    candidates = retriever.retrieve(question, top_k=20)
    final = reranker.rerank(question, candidates)

    # Generate
    result = generator.generate(question, final)

    print(f"\nA: {result.answer}")

    if result.citations:
        print("\nSources:")
        for c in result.citations:
            print(f"  • {c.get('company')} — {c.get('document')}, p.{c.get('page')}")

    if result.verification:
        print(f"\nNumerical check: {result.verification.summary}")
        if not result.verification.verified:
            for m in result.verification.mismatches:
                print(f"  ⚠️  {m.message}")

demo_questions = [
    "What was Tesla's total revenue for FY2023?",
    "How did Tesla's gross margin change from FY2022 to FY2023?",
    "What are the key risk factors Tesla identifies in its 10-K?",
    "What was Google's revenue in FY2023?",  # out-of-scope
]

for q in demo_questions:
    ask(q)